## FINLORA FINANCE FRAUD DETECTION & SCORING PROJECT ANALYSIS

## Business Prospects — 
Finlora a fintech company that provide transaction monitoring services to clients and merchandise. Due to increase data injection, their fixed monitoring threshold has failed to handle fraud transactions from legitimate transactions. A risk-scoring model built to detect fraud transaction from legitimate payroll, and device priority alerts to flag fraud transaction urgency regardless of actual risk.

## Business Problem — 
Replace rule-based monitoring missing subtle fraud as volume grows with a validated model that catches true fraud transactions.
Translating all information’s into Data driven task— binary classification of transactions as fraudulent vs. legitimate, with a probability score for prioritization.


## Analytical Framework:

| SN | NoteBook Outline |
|---|---|
| **1** | Problem definition document & data source inventory |
| **2** | Clean, analysis-ready dataset (transactions + accounts, joined and feature-engineered) |
| **3** | EDA summary report (fraud vs. legitimate patterns)|
| **4** | Feature importance / behavioral drivers document|
| **5** | Two trained models: Logistic Regression (baseline) + Random Forest (primary)|
| **6** | Model evaluation report (precision, recall, F1, ROC-AUC, confusion matrix, threshold analysis)|
| **7** | Deployed Streamlit dashboard: prioritized, explainable fraud review queue
| **8** | Final documentation + handover summary

## IMPORTING LIBRARIES

In [132]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              confusion_matrix, roc_curve, precision_recall_curve,
                              classification_report)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 60)
RANDOM_STATE = 42

## LOAD DATASETS

In [133]:
transaction_data = pd.read_csv(
    r"C:\Users\User\Documents\AMDARI_IMS\Finlora_Finance_PJ\Dataset\Raw_data\finlora_transactions.csv")
account_data = pd.read_csv(
    r"C:\Users\User\Documents\AMDARI_IMS\Finlora_Finance_PJ\Dataset\Raw_data\finlora_accounts.csv")

print(f"Raw transactions: {len(transaction_data):,} rows, {transaction_data.shape[1]} columns")
print(f"Raw accounts:     {len(account_data):,} rows, {account_data.shape[1]} columns")
transaction_data.head()

Raw transactions: 126,000 rows, 24 columns
Raw accounts:     7,200 rows, 8 columns


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,channel,amount,currency,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,Card Not Present,84233.00,NGN,9.21,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,Web Dashboard,9143.33,NGN,1.00,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,Mobile App,20639.00,NGN,2.26,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,Card Not Present,3177.43,NGN,0.21,14891.16,0,NG,NG,0,NaN,NaN,974,Completed,0
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,Mobile App,97839.52,NGN,1.97,49609.59,0,NG,NG,0,DEV-9055235108,0.0,1035,Declined,0


### INVESTING DATASETS

In [134]:
print("Nulls per column (transactions):")
print(transaction_data.isnull().sum()[transaction_data.isnull().sum() > 0])
print("-"*40)
print("Duplicate transaction_id:", transaction_data['transaction_id'].duplicated().sum())
print("Duplicate account_id in accounts:", account_data['account_id'].duplicated().sum())
print("Negative amounts:", (transaction_data['amount'] < 0).sum())
print("avg_transaction_amount_30d <= 0:", (transaction_data['avg_transaction_amount_30d'] <= 0).sum())
print("-"*67)

# investigating if amount_to_avg_ratio actually equal |amount| / avg_transaction_amount_30d?
check_ratio = transaction_data['amount'].abs() / transaction_data['avg_transaction_amount_30d']
mismatch = (check_ratio - transaction_data['amount_to_avg_ratio']).abs() > 0.5
print(f"Rows where amount_to_avg_ratio is inconsistent with amount/avg: {mismatch.sum():,}")


Nulls per column (transactions):
merchant_name    28610
device_id        14334
is_new_device    14334
dtype: int64
----------------------------------------
Duplicate transaction_id: 0
Duplicate account_id in accounts: 0
Negative amounts: 291
avg_transaction_amount_30d <= 0: 147
-------------------------------------------------------------------
Rows where amount_to_avg_ratio is inconsistent with amount/avg: 265


##### Data Cleanning: (Missing Values, negative values, duplicates)

In [135]:
# ----------------------------------------------------------
# 1. Check missing values before treatment
# ----------------------------------------------------------
'''
Create a dataframe to show total number of missing values
and percentage of missing values in transaction data
'''

missing_data = pd.DataFrame({"Missing Data": transaction_data.isnull().sum(), "Missing Percentage": (transaction_data.isnull().mean() * 100).round(2)})

# Show only columns that contain missing values
missing_data = (missing_data[missing_data["Missing Data"] > 0].sort_values("Missing Percentage", ascending=False))

print("Missing values BEFORE treatment:")
print("--"*25)
print(missing_data)
print()


# Visualizing the Missing Data
print("Visaulization of Missing values BEFORE treatment:")
print("--"*25)
fig = px.bar(missing_data, x=missing_data.index, y="Missing Percentage",
             title="Missing Data Percentage by Feature",
             labels={"x": "Features", "Missing Percentage": "Missing Percentage (%)"},
             text="Missing Percentage")

fig.update_traces( texttemplate="%{text:.2f}%", textposition="outside")

fig.update_layout(showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
                  xaxis=dict(showgrid=False, tickangle=-45), yaxis=dict(showgrid=False))

fig.show()

Missing values BEFORE treatment:
--------------------------------------------------
               Missing Data  Missing Percentage
merchant_name         28610               22.71
device_id             14334               11.38
is_new_device         14334               11.38

Visaulization of Missing values BEFORE treatment:
--------------------------------------------------


In [136]:
# create a clean data log container for the cleaning process
cleaning_log = []
tx = transaction_data.copy()
acc = account_data.copy()

# Handling Duplicates.
dup_tx, dup_id = tx.duplicated().sum(), tx['transaction_id'].duplicated().sum() # referening with unique data transaction_id identifier
cleaning_log.append(f"Duplicate full rows: {dup_tx}; duplicate transaction_id: {dup_id}")
tx = tx.drop_duplicates(subset='transaction_id')

dup_acc = acc['account_id'].duplicated().sum()   # referening with unique data account_id identifier
cleaning_log.append(f"Duplicate account_id: {dup_acc}")
acc = acc.drop_duplicates(subset='account_id')

# Handle missing merchant_name: Payroll/P2P transfers have no merchant ---
n_missing_merchant = tx['merchant_name'].isnull().sum()
cleaning_log.append(f"merchant_name missing: {n_missing_merchant:,} rows (Payroll/P2P transfers) -> filled 'N/A (transfer)'")
tx['merchant_name'] = tx['merchant_name'].fillna('N/A (transfer)')

# Handle missing device_id & is_new_device 
n_missing_device = tx['device_id'].isnull().sum()
cleaning_log.append(f"device_id/is_new_device missing: {n_missing_device:,} rows ({n_missing_device/len(tx):.1%}) -> device_id='UNKNOWN', is_new_device=0")
tx['device_id'] = tx['device_id'].fillna('UNKNOWN')
tx['is_new_device'] = tx['is_new_device'].fillna(0).astype(int)

##----------Cleaning Negative Input Data----------##

# Handling negative amounts: as genuine refunds 
n_negative = (tx['amount'] < 0).sum()
cleaning_log.append(f"Negative amounts: {n_negative} (all Retail, consistent with refunds) -> flagged with is_refund")
tx['is_refund'] = (tx['amount'] < 0).astype(int)

# Handling Negative/Invalid 30-day average baseline 
n_bad_avg = (tx['avg_transaction_amount_30d'] <= 0).sum()
cleaning_log.append(f"avg_transaction_amount_30d <= 0: {n_bad_avg} rows -> replaced with account/global median positive value")
acct_median_avg = tx.loc[tx['avg_transaction_amount_30d'] > 0].groupby('account_id')['avg_transaction_amount_30d'].median()
global_median_avg = tx.loc[tx['avg_transaction_amount_30d'] > 0, 'avg_transaction_amount_30d'].median()
bad_mask = tx['avg_transaction_amount_30d'] <= 0
tx.loc[bad_mask, 'avg_transaction_amount_30d'] = tx.loc[bad_mask, 'account_id'].map(acct_median_avg).fillna(global_median_avg)

# Recompute amount_to_avg_ratio cleanly 
cleaning_log.append("amount_to_avg_ratio recomputed as |amount| / avg_transaction_amount_30d (source column was inconsistent)")
tx['amount_to_avg_ratio'] = (tx['amount'].abs() / tx['avg_transaction_amount_30d']).round(4)


# Handle categorical data causing inconsistency when merging both dataset
for c in ['account_type','kyc_tier','merchant_category','channel','status','currency','day_of_week']:
    tx[c] = tx[c].str.strip()
cleaning_log.append("Stripped whitespace on categorical columns; no case-inconsistent duplicate labels found")

# Handle timestamps
tx['timestamp'] = pd.to_datetime(tx['timestamp'])
cleaning_log.append("Parsed timestamp to datetime; all values valid")



###---------Concatenating both dataset--------###
acc['account_created_date'] = pd.to_datetime(acc['account_created_date'])
finlora_data = tx.merge(acc[['account_id','account_holder_name','account_created_date','personal_spend_baseline_usd']],
              on='account_id', how='left')
n_unmatched = finlora_data['account_created_date'].isnull().sum()
cleaning_log.append(f"Transactions with no matching account after merge: {n_unmatched}")
cleaning_log.append(f"Final analysis-ready dataset: {len(finlora_data):,} rows, {finlora_data.shape[1]} columns")


###--------Cleaned Dateset Summary--------###
for line in cleaning_log:
    print("=>", line)


=> Duplicate full rows: 0; duplicate transaction_id: 0
=> Duplicate account_id: 0
=> merchant_name missing: 28,610 rows (Payroll/P2P transfers) -> filled 'N/A (transfer)'
=> device_id/is_new_device missing: 14,334 rows (11.4%) -> device_id='UNKNOWN', is_new_device=0
=> Negative amounts: 291 (all Retail, consistent with refunds) -> flagged with is_refund
=> avg_transaction_amount_30d <= 0: 147 rows -> replaced with account/global median positive value
=> amount_to_avg_ratio recomputed as |amount| / avg_transaction_amount_30d (source column was inconsistent)
=> Stripped whitespace on categorical columns; no case-inconsistent duplicate labels found
=> Parsed timestamp to datetime; all values valid
=> Transactions with no matching account after merge: 0
=> Final analysis-ready dataset: 126,000 rows, 28 columns


### Save New Dataset

In [137]:
finlora_data.to_csv(r'C:\Users\User\Documents\AMDARI_IMS\Finlora_Finance_PJ\Dataset\preprocessed_data\cleaned_finlora_data.csv', index=False)
finlora_data.head()
# Data Summary
finlora_summary = pd.DataFrame({
    "Data Type": finlora_data.dtypes.astype(str),
    "Missing Values": finlora_data.isna().sum(),
    "% Missing Values": (finlora_data.isna().mean() * 100).round(2),
    "Unique Values": finlora_data.nunique(),
    "Duplicate Values": [finlora_data[col].duplicated().sum()for col in finlora_data.columns]})
finlora_summary = (finlora_summary.sort_values("% Missing Values", ascending=False))

print("Dataset Shape:", finlora_data.shape)
print("Duplicate Rows:", finlora_data.duplicated().sum())

finlora_summary

Dataset Shape: (126000, 28)
Duplicate Rows: 0


,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
transaction_id,object,0,0.0,126000,0
account_id,object,0,0.0,7144,118856
account_created_date,datetime64[ns],0,0.0,1441,124559
account_holder_name,object,0,0.0,1000,125000
is_refund,int32,0,0.0,2,125998
is_fraud,int64,0,0.0,2,125998
status,object,0,0.0,3,125997
account_age_days,int64,0,0.0,1540,124460
is_new_device,int32,0,0.0,2,125998
device_id,object,0,0.0,14932,111068
